# Lab 2 — Gemini API com Python

Neste notebook você vai transformar um **System Prompt** em uma pequena aplicação Python que conversa com o Gemini pela API.

## 0. Antes de começar

Você precisará de uma chave da Gemini API.

1. Acesse o [Google AI Studio](https://aistudio.google.com/).
2. Abra **API Keys** e crie uma chave.
3. No Colab, abra **Secrets** na barra lateral.
4. Crie um secret chamado `GEMINI_API_KEY`.
5. Cole sua chave e habilite o acesso para este notebook.

> **Nunca escreva sua API key diretamente em uma célula que será compartilhada.**

## 1. Instale o SDK

Vamos usar o SDK oficial `google-genai`.

O `-q` reduz as mensagens mostradas pelo `pip` e o `-U` solicita uma versão atualizada da biblioteca.

In [ ]:
!pip install -q -U "google-genai>=2.3.0"

## 2. Crie o cliente da API

O código abaixo recupera a chave armazenada nos Secrets do Colab e cria um cliente.

A variável `client` será usada nas próximas chamadas.

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Cliente criado com sucesso.")

## 3. Primeira chamada

Comece com uma única mensagem.

Leia o código antes de executar e identifique:

- qual modelo está sendo usado;
- qual texto será enviado;
- onde a resposta será recuperada.

In [ ]:
MODEL = "gemini-3.5-flash-lite"

interaction = client.interactions.create(
    model=MODEL,
    input="Explique em uma frase o que é uma API."
)

print(interaction.output_text)

### Observe a resposta da API

Além do texto, a interação possui um identificador.

Ele será importante mais adiante.

In [ ]:
print("ID da interação:", interaction.id)

### Sua vez

Altere a mensagem abaixo e faça uma pergunta diferente.

Sugestão:

`Explique a diferença entre API e biblioteca usando uma analogia.`

In [ ]:
minha_pergunta = "ESCREVA SUA PERGUNTA AQUI"

interaction_teste = client.interactions.create(
    model=MODEL,
    input=minha_pergunta
)

print(interaction_teste.output_text)

## 4. Transformando o modelo em um assistente

Uma chamada isolada gera texto, mas ainda não define claramente o papel da nossa aplicação.

Vamos usar um **System Prompt** para criar a NIA, assistente da TechStore.

In [ ]:
SYSTEM_PROMPT = """
Você é a NIA, assistente virtual da TechStore, uma loja fictícia
de eletrônicos criada para uma atividade acadêmica.

Seu objetivo é ajudar clientes com dúvidas simples sobre produtos
e problemas básicos de uso.

Regras:
- responda em português;
- seja objetiva e educada;
- faça perguntas quando faltarem informações;
- não invente preços, estoque, informações de pedidos ou dados de clientes;
- informe claramente que você não possui acesso ao sistema de pedidos;
- nunca peça senhas, números completos de cartão ou outros dados sensíveis;
- se não souber a resposta, diga que não possui informação suficiente;
- quando o problema exigir análise física do equipamento, recomende atendimento técnico;
- responda normalmente em até 120 palavras.

Informações disponíveis:
- o suporte humano funciona de segunda a sexta, das 9h às 18h;
- a TechStore oferece garantia fictícia de 12 meses contra defeitos
  de fabricação para os produtos deste laboratório;
- você não possui acesso a preços em tempo real, estoque ou situação de pedidos.
"""

Agora enviaremos uma mensagem usando `system_instruction`.

In [ ]:
interaction = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT,
    input="Meu pedido 58421 ainda não chegou. Onde ele está?"
)

print("NIA:", interaction.output_text)

### Pare e observe

A NIA deveria evitar inventar a situação do pedido porque o System Prompt diz que ela não possui acesso ao sistema de pedidos.

Agora faça outro teste.

In [ ]:
pergunta = "Meu notebook não está ligando. O que posso verificar?"

interaction = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT,
    input=pergunta
)

print("NIA:", interaction.output_text)

## 5. Recebendo mensagens com `input()`

Até agora, as mensagens estavam escritas dentro do código.

Vamos permitir que a pessoa que estiver executando o notebook digite a mensagem.

In [ ]:
mensagem = input("Você: ")

interaction = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT,
    input=mensagem
)

print("NIA:", interaction.output_text)

## 6. O modelo lembra da conversa?

Vamos testar primeiro **duas chamadas independentes**.

Na primeira, informaremos o modelo do computador.

Na segunda, perguntaremos qual é o modelo.

Observe que a segunda chamada não aponta para a primeira.

In [ ]:
primeira = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT,
    input="Meu computador é um notebook Orion 14."
)

print("NIA 1:", primeira.output_text)

segunda = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT,
    input="Qual é o modelo do meu computador?"
)

print("NIA 2:", segunda.output_text)

### O que aconteceu?

Cada chamada acima foi criada como uma interação independente.

Agora faremos a segunda chamada apontar para a primeira usando:

```python
previous_interaction_id=primeira.id
```

Isso informa à API que a nova mensagem continua a conversa anterior.

In [ ]:
primeira = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT,
    input="Meu computador é um notebook Orion 14."
)

print("NIA 1:", primeira.output_text)

segunda = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT,
    previous_interaction_id=primeira.id,
    input="Qual é o modelo do meu computador?"
)

print("NIA 2:", segunda.output_text)

### Um detalhe importante

`previous_interaction_id` mantém o histórico da conversa.

Porém, o `system_instruction` é uma configuração da interação atual. Por isso, nós o enviaremos novamente a cada nova rodada do nosso assistente.

Esse detalhe fica escondido quando usamos uma interface de chat pronta. No código, a aplicação precisa controlar esse comportamento.

## 7. Construa uma conversa de três rodadas

Complete a terceira mensagem e verifique se o contexto continua funcionando.

In [ ]:
rodada_1 = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT,
    input="Meu notebook é um Orion 14 e ele não está ligando."
)

print("NIA:", rodada_1.output_text)

rodada_2 = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT,
    previous_interaction_id=rodada_1.id,
    input="A luz do carregador está acesa."
)

print("NIA:", rodada_2.output_text)

terceira_mensagem = "ESCREVA AQUI UMA CONTINUAÇÃO DA CONVERSA"

rodada_3 = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT,
    previous_interaction_id=rodada_2.id,
    input=terceira_mensagem
)

print("NIA:", rodada_3.output_text)

## 8. Aplicação final

Agora vamos juntar as ideias anteriores em um `while`.

A aplicação continuará rodando até que o usuário digite `sair`.

Leia o código antes de executá-lo.

In [ ]:
previous_interaction_id = None

print("NIA: Olá! Sou a assistente virtual da TechStore.")
print("Digite 'sair' para encerrar.\n")

while True:
    mensagem = input("Você: ").strip()

    if mensagem.lower() == "sair":
        print("NIA: Atendimento encerrado.")
        break

    if not mensagem:
        continue

    argumentos = {
        "model": MODEL,
        "system_instruction": SYSTEM_PROMPT,
        "input": mensagem
    }

    if previous_interaction_id is not None:
        argumentos["previous_interaction_id"] = previous_interaction_id

    interaction = client.interactions.create(**argumentos)

    previous_interaction_id = interaction.id

    print("NIA:", interaction.output_text)
    print()

## 9. Testes obrigatórios

Execute novamente a aplicação final e realize estes testes.

| Teste | Entrada | O que observar |
| --- | --- | --- |
| Normal | `Meu notebook não está ligando. O que posso verificar?` | A resposta é útil e objetiva? |
| Dado inexistente | `Meu pedido 58421 está onde?` | A NIA evita inventar informações? |
| Contexto | `Meu computador é um Orion 14.` e depois `Qual é o modelo dele?` | A segunda mensagem usa o contexto? |

Depois crie **um quarto teste por conta própria**.

Registre abaixo o que você testou e o que aconteceu.

### Registro dos testes

**Teste 1**

Resultado:

**Teste 2**

Resultado:

**Teste 3**

Resultado:

**Teste 4 criado por você**

Entrada:

Resultado:

**A NIA deixou de seguir alguma regra? Se sim, qual?**

## 10. Desafio

Escolha **uma** opção.

### A — Melhorar a experiência

Faça a aplicação reconhecer `sair`, `fim` e `encerrar`, além de mostrar uma mensagem de despedida.

### B — Refinar o comportamento

Altere o System Prompt para que a NIA:

1. faça apenas uma pergunta de diagnóstico por vez;
2. use passos numerados quando explicar um procedimento;
3. encaminhe para suporte humano quando a situação não puder ser resolvida com as informações disponíveis.

### C — Criar outra aplicação

Mantenha a estrutura Python e substitua a TechStore por outro assistente.

Exemplos: estudos, biblioteca, cafeteria, hotel ou jogo textual.

**Pergunta para refletir:** quanto do código precisou mudar?

## 11. Espaço para sua implementação do desafio

Use a célula abaixo.

In [ ]:
# Implemente aqui a opção A, B ou C do desafio.

## 12. Extra — inspecionando o uso de tokens

As entradas, o System Prompt e a resposta são convertidos em tokens.

Se houver uma interação disponível na variável `interaction`, você pode observar o uso:

In [ ]:
print("Tokens de entrada:", interaction.usage.total_input_tokens)
print("Tokens de saída:", interaction.usage.total_output_tokens)
print("Tokens totais:", interaction.usage.total_tokens)